In [ ]:
import pandas as pd

train_df = pd.read_csv("/content/drive/MyDrive/project/train.csv")
test_df  = pd.read_csv("/content/drive/MyDrive/project/test.csv")

train_df["input_text"] = train_df["title"].fillna("") + " " + train_df["text"].fillna("")
test_df["input_text"]  = test_df["title"].fillna("") + " " + test_df["text"].fillna("")

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("/content/drive/MyDrive/project/tokenizer")

In [ ]:
train_encodings = tokenizer(
    list(train_df["input_text"]),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    list(test_df["input_text"]),
    truncation=True,
    padding=True,
    max_length=256
)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_df["label"])
test_dataset  = NewsDataset(test_encodings,  test_df["label"])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=8)

In [ ]:
import torch.nn as nn

class CapsuleLayer(nn.Module):
    def __init__(self, input_dim=768, num_capsules=8, dim_capsule=16):
        super().__init__()
        self.num_capsules = num_capsules
        self.dim_capsule = dim_capsule
        self.fc = nn.Linear(input_dim, num_capsules * dim_capsule)

    def forward(self, x):
        x = self.fc(x)
        x = x.view(-1, self.num_capsules, self.dim_capsule)

        norm = torch.norm(x, dim=-1, keepdim=True)
        x = (x / (1 + norm**2)) * (norm / (1 + norm))

        return x

In [ ]:
from transformers import BertModel

class BertCapsuleModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.bert = BertModel.from_pretrained(
            "/content/drive/MyDrive/project/bert_model"
        )

        self.capsule = CapsuleLayer(768, 8, 16)

        self.dropout = nn.Dropout(0.3)

        # 🔥 FINAL CLASSIFIER (VERY IMPORTANT)
        self.classifier = nn.Linear(8 * 16, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        cls = outputs.last_hidden_state[:, 0, :]

        x = self.capsule(cls)
        x = x.view(x.size(0), -1)

        x = self.dropout(x)

        return self.classifier(x)

In [ ]:

#this is training setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertCapsuleModel().to(device)

# Freeze BERT (phase 1)
for param in model.bert.parameters():
    param.requires_grad = False

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
criterion = nn.CrossEntropyLoss()



Loading weights:   0%|          | 0/199 [00:01<?, ?it/s]

BertModel LOAD REPORT from: /content/drive/MyDrive/project/bert_model
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Train Capsule

import os

save_path = "/content/drive/MyDrive/project/capsule_model"
os.makedirs(save_path, exist_ok=True)

for epoch in range(3):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

    torch.save(model.state_dict(), f"{save_path}/phase1_epoch_{epoch+1}.pth")


Epoch 1, Loss: 0.3820
Epoch 2, Loss: 0.3532
Epoch 3, Loss: 0.3475


In [ ]:
# Fine Tune Bert


# Unfreeze BERT

for param in model.bert.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

for epoch in range(2):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Fine-tune Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# Save final model
torch.save(model.state_dict(), f"{save_path}/final_model.pth")

print(" Final model saved!")

Fine-tune Epoch 1, Loss: 0.2981
Fine-tune Epoch 2, Loss: 0.2818
🔥 Final model saved!


In [ ]:
# Checking Models Accuracy And F1

from sklearn.metrics import accuracy_score, f1_score

model.eval()

preds_list = []
labels_list = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask)
        preds = torch.argmax(outputs, dim=1)

        preds_list.extend(preds.cpu().numpy())
        labels_list.extend(labels.cpu().numpy())

acc = accuracy_score(labels_list, preds_list)
f1  = f1_score(labels_list, preds_list, average="weighted")

print(f"🔥 Test Accuracy: {acc:.4f}")
print(f"🔥 Test F1 Score: {f1:.4f}")

🔥 Test Accuracy: 0.8949
🔥 Test F1 Score: 0.8946


In [ ]:
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===============================
# 🔹 LOAD MODEL (IMPORTANT)
# ===============================
model_path = "/content/drive/MyDrive/project/capsule_model/final_model.pth"

model = BertCapsuleModel().to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print("Model Loaded Successfully")


# ===============================
#  EVALUATION FUNCTION
# ===============================
def evaluate(model, loader):
    model.eval()

    preds_list = []
    labels_list = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask)
            preds = torch.argmax(outputs, dim=1)

            preds_list.extend(preds.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())

    acc = accuracy_score(labels_list, preds_list)
    f1  = f1_score(labels_list, preds_list, average="weighted")

    return acc, f1, labels_list, preds_list


# ===============================
# 🔹 TRAIN ACCURACY
# ===============================
train_acc, train_f1, _, _ = evaluate(model, train_loader)

print("\n🔥 Training Results")
print(f"Accuracy: {train_acc:.4f}")
print(f"F1 Score: {train_f1:.4f}")


# ===============================
# 🔹 TEST ACCURACY
# ===============================
test_acc, test_f1, labels, preds = evaluate(model, test_loader)

print("\n🔥 Test Results")
print(f"Accuracy: {test_acc:.4f}")
print(f"F1 Score: {test_f1:.4f}")



# 🔹 DETAILED REPORT

print("\n📊 Classification Report:")
print(classification_report(labels, preds))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /content/drive/MyDrive/project/bert_model
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model Loaded Successfully

🔥 Training Results
Accuracy: 0.9045
F1 Score: 0.9043

🔥 Test Results
Accuracy: 0.8949
F1 Score: 0.8946

📊 Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.85      0.89      6266
           1       0.86      0.94      0.90      6444

    accuracy                           0.89     12710
   macro avg       0.90      0.89      0.89     12710
weighted avg       0.90      0.89      0.89     12710

